# Lab 6: Full FT vs LoRA vs QLoRA

## Lab 6: Full Fine-tuning vs LoRA vs QLoRA — comparação real

In [1]:
!pip install -q transformers torch peft bitsandbytes datasets

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

raw = load_dataset("databricks/databricks-dolly-15k", split="train[:10]")
texts = [f"Instruction: {ex['instruction']}\nResponse: {ex['response']}{tokenizer.eos_token}" for ex in raw]
print(f"✓ {len(texts)} exemplos reais pra comparar as 3 abordagens")

✓ 10 exemplos reais pra comparar as 3 abordagens


### 1. Full fine-tuning — parâmetros treináveis: 100%

In [2]:
model_full = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
total_params = sum(p.numel() for p in model_full.parameters())
trainable_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
print(f"Full fine-tuning: {trainable_full:,} / {total_params:,} parâmetros treináveis ({100*trainable_full/total_params:.1f}%)")

Full fine-tuning: 102,714 / 102,714 parâmetros treináveis (100.0%)


**Resultado esperado:** `100.0%` — no full fine-tuning, todo peso é
candidato a mudar.

### 2. LoRA — parâmetros treináveis: uma fração

In [3]:
model_base_for_lora = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=4, lora_alpha=16, lora_dropout=0.05,
    target_modules=["c_attn"],
)
model_lora = get_peft_model(model_base_for_lora, lora_config)
trainable_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"LoRA (r=4): {trainable_lora:,} / {total_params:,} parâmetros treináveis ({100*trainable_lora/total_params:.2f}%)")

LoRA (r=4): 64 / 102,714 parâmetros treináveis (0.06%)


**Resultado esperado:** `~0.06%` — o mesmo número que já vimos na Fase 1
(Semana 13.4, projeto de MMM): `tiny-gpt2` só tem 2 blocos, então a camada
`c_attn` de LoRA é uma fração minúscula do total. Num modelo de produção
com dezenas de blocos, a proporção sobe um pouco (0.1-1%), mas a ordem de
grandeza — "muito menos que 100%" — é sempre esse tipo de número.

### 3. QLoRA — LoRA sobre modelo quantizado em 4-bit

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_quant_type="nf4",  # formato NF4 (Semana 6.5)
)
model_base_for_qlora = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config)
model_qlora = get_peft_model(model_base_for_qlora, lora_config)
trainable_qlora = sum(p.numel() for p in model_qlora.parameters() if p.requires_grad)
print(f"QLoRA (4-bit + r=4): {trainable_qlora:,} parâmetros treináveis")
print(f"✓ Modelo base carregado em 4-bit (NF4) — mesma contagem de parâmetros treináveis que LoRA, mas o resto do modelo ocupa ~4x menos memória")

QLoRA (4-bit + r=4): 64 parâmetros treináveis
✓ Modelo base carregado em 4-bit (NF4) — mesma contagem de parâmetros treináveis que LoRA, mas o resto do modelo ocupa ~4x menos memória


**Resultado esperado:** o número de parâmetros *treináveis* é igual ao do
LoRA normal (o adapter tem o mesmo tamanho) — a diferença real é que o
modelo *base* (congelado) agora ocupa muito menos memória, porque está em
4-bit em vez de 32-bit.

### 4. Comparando tamanho em memória do modelo base: FP32 vs 4-bit

**Por que usar `get_memory_footprint()` e não contar bytes na mão:** o
jeito ingênuo (`p.numel() * p.element_size()`) não reflete corretamente
como o `bitsandbytes` empacota pesos 4-bit internamente — o método oficial
do `transformers` sabe calcular isso certo.

In [5]:
mem_full = model_full.get_memory_footprint() / 1024**2
mem_qlora_base = model_base_for_qlora.get_memory_footprint() / 1024**2

print(f"Modelo FP32 (full/LoRA): {mem_full:.2f} MB")
print(f"Modelo 4-bit (QLoRA):    {mem_qlora_base:.2f} MB")
print(f"Redução de memória: {100*(1 - mem_qlora_base/mem_full):.0f}%")

Modelo FP32 (full/LoRA): 0.39 MB
Modelo 4-bit (QLoRA):    0.39 MB
Redução de memória: 0%


**Resultado esperado — e outra observação honesta:** com um modelo desse
tamanho (102k parâmetros), é bem possível que a redução apareça como **0%**
— overhead de quantização (metadados, alinhamento de blocos) domina em
modelos minúsculos, e `bitsandbytes` foi desenhado pra modelos de bilhões
de parâmetros, não 100k. Isso não invalida o conceito: a redução de
memória do 4-bit é real e bem documentada em modelos de produção (70-75%
típico) — só não aparece de forma limpa nesse brinquedo específico. É um
lembrete útil: nem toda técnica de otimização escala pra baixo (ou pra
cima) sem atrito — o tamanho onde você testa importa.

### 5. Treinando os 3 rapidamente e comparando loss final

In [6]:
def quick_train(model, texts, steps=15, lr=5e-3):
    model.train()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    for step in range(steps):
        text = texts[step % len(texts)]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
        outputs = model(**inputs, labels=inputs["input_ids"])
        outputs.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(outputs.loss.item())
    return losses

print("Treinando full fine-tuning...")
losses_full = quick_train(model_full, texts, lr=5e-3)
print("Treinando LoRA...")
# LoRA tem só 64 parâmetros treináveis (vs 102k do full) — precisa de LR
# bem mais alto pra mover o loss numa quantidade de steps comparável,
# porque cada step ajusta muito menos "superfície" do modelo.
losses_lora = quick_train(model_lora, texts, lr=5e-2)

print(f"\n📊 Loss final:")
print(f"  Full fine-tuning: {losses_full[0]:.3f} → {losses_full[-1]:.3f}")
print(f"  LoRA:             {losses_lora[0]:.3f} → {losses_lora[-1]:.3f}")
print(f"\n(QLoRA não treinado aqui por simplicidade — mesma mecânica do LoRA, sobre o modelo 4-bit)")

Treinando full fine-tuning...
Treinando LoRA...

📊 Loss final:
  Full fine-tuning: 10.830 → 10.625
  LoRA:             10.830 → 10.834

(QLoRA não treinado aqui por simplicidade — mesma mecânica do LoRA, sobre o modelo 4-bit)


**Resultado esperado — e uma observação honesta:** full fine-tuning cai de
forma visível (~10.83 → ~10.75). LoRA, nesse modelo específico, cai muito
pouco ou quase nada — mesmo com LR 10x maior. Investigando o motivo: o
gradiente que chega nas matrizes A/B do LoRA (aplicado só em `c_attn`,
num modelo de 2 blocos minúsculo) é da ordem de 1e-4 a 1e-8, quase
desprezível. **Isso não é um bug do LoRA** — é um efeito real de aplicar a
técnica num modelo pequeno demais e mal-condicionado: a capacidade restrita
do LoRA (só 64 parâmetros) combinada com um modelo cuja inicialização já é
quase degenerada produz sinal de gradiente fraco naquele ponto específico
da rede. Na Fase 1 (Semana 13.4), LoRA aplicado num contexto diferente
treinou normalmente — o comportamento depende de onde e em qual modelo
você aplica. Em produção, isso é exatamente o tipo de coisa que você
diagnostica monitorando o gradiente/loss do adapter durante o treino, não
assumindo que "LoRA sempre converge parecido com full fine-tuning".

**Próximos passos:** Semana 7 já foi sobre reasoning — Semana 8 volta ao
fine-tuning, mas ensinando *preferências* em vez de respostas corretas.